# Notebook 4: Orchestrating Spark with Apache Airflow & DWH Ops Playbook (Student Lab)
### Hands-on Workshop: Apache Spark Foundation & Ingestion Framework (Day 2 Afternoon)
### Related Presentation Slides: Slides 23 - 25 (Module 7: Orchestrating Spark with Airflow)

---

## Learning Objectives:
1. Understand the distinct boundary: Spark (Compute Engine) vs. Airflow (Conductor/Orchestrator) (Slide 23).
2. The Golden Rule of Airflow: Never transfer DataFrames via XCom (Slide 23).
3. Anatomy of a Production DAG: `default_args`, `catchup=False`, `S3KeySensor`, `BashOperator`, `PythonOperator` (Slide 24).
4. Live Execution in Colab: Test and run the DAG in-process using `dag.test()`.
5. DWH Ops Incident Recovery: Grid View, reading container logs, and the Clear Task action (Slide 25).


In [ ]:
# Install Apache Airflow and PySpark in Colab
!pip install -q apache-airflow pyspark

import airflow
print(f"Apache Airflow version: {airflow.__version__} installed successfully.")


---
## Step 1: Spark vs. Airflow — System Responsibilities (Related: Slide 23)

| Role | Tool | Responsibility |
| :--- | :--- | :--- |
| Compute Engine | Apache Spark | Executes heavy computations in RAM (100M rows, joins, complex math). |
| Workflow Orchestrator | Apache Airflow | Manages workflow order, checks prerequisites (`Sensor`), triggers Spark (`SparkSubmitOperator`), manages retries, alerts. |

### The Golden Rule (Slide 23):
> Airflow backend is a relational metadata database (PostgreSQL/MySQL).
> If you push large DataFrames through Airflow XCom, you risk crashing the metadata DB.
> Data stays in Cloud Storage (S3) & Spark RAM; Airflow passes only paths, dates, and status codes.


---
## Step 2: Defining the Production Spark DAG (Related: Slide 24)

Let's inspect and construct the DAG definition in Python:


In [ ]:
from datetime import datetime, timedelta
import os
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator

# 1. Default Arguments (SLA, Retries, Alerts)
default_args = {
    "owner": "dwh_ops",
    "depends_on_past": False,
    "email": ["dwh-alerts@company.com"],
    "email_on_failure": True,
    "email_on_retry": False,
    "retries": 2,                           # Retry up to 2 times on transient failures
    "retry_delay": timedelta(seconds=10),   # Wait 10s between retries (short for lab demo)
    "execution_timeout": timedelta(hours=1),
}

# 2. DAG Definition
dag = DAG(
    dag_id="spark_bundesliga_ingestion_pipeline",
    default_args=default_args,
    description="Orchestrate daily Bundesliga stats ingestion into S3 Parquet",
    schedule_interval="0 2 * * *",          # Daily at 02:00 AM UTC
    start_date=datetime(2026, 1, 1),
    catchup=False,                          # Important: avoid catchup in production!
    max_active_runs=1,
    tags=["spark", "dwh", "bundesliga"]
)

# Tasks:

# Task 1: Check upstream raw file arrival (Simulation of S3KeySensor)
def check_raw_incoming_file(**context):
    file_path = "data/raw/bundesliga_events.json"
    print(f"[Sensor] Checking if raw file exists: {file_path}...")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing upstream data: {file_path}")
    print("[Sensor] Upstream data is present. Triggering Spark compute.")
    return True

wait_for_raw_data = PythonOperator(
    task_id="wait_for_raw_data",
    python_callable=check_raw_incoming_file,
    dag=dag
)

# Task 2: Execute PySpark Ingestion Job
run_spark_job = BashOperator(
    task_id="run_spark_job",
    bash_command="python -c 'print(\"[Spark Worker] PySpark Job executed successfully. Output written to Parquet.\")'",
    dag=dag
)

# Task 3: Metadata / Partition Quality Validation
def validate_output_partitions(**context):
    print("[Validator] Validating target Parquet partitions...")
    parquet_path = "data/processed/bundesliga_parquet"
    if os.path.exists(parquet_path):
        print(f"[Validator] Partition directory verified: {parquet_path}")
    else:
        print("[Validator] Simulation mode: partition check passed.")
    return "VALIDATED"

validate_target = PythonOperator(
    task_id="validate_target",
    python_callable=validate_output_partitions,
    dag=dag
)

# Task 4: Downstream Notification
notify_bi = BashOperator(
    task_id="notify_bi",
    bash_command="echo '[Alert] Ingestion completed. PowerBI semantic model refresh triggered.'",
    dag=dag
)

# Define Dependency Graph
wait_for_raw_data >> run_spark_job >> validate_target >> notify_bi

print(f"DAG '{dag.dag_id}' compiled with {len(dag.tasks)} tasks.")


---
## Step 3: Live DAG Testing in Colab with dag.test()

In Apache Airflow 2.x, `dag.test()` executes all tasks in order within the current Python process.
Watch the live execution output below:


In [ ]:
# Run the DAG live
print("--- STARTING LIVE AIRFLOW PIPELINE RUN ---")
dag.test(execution_date=datetime(2026, 9, 10))
print("--- PIPELINE COMPLETED ---")


---
## Step 4: DWH Ops Incident Recovery Playbook (Related: Slide 25)

What happens when an alert triggers during night operations?

```
[DAG Grid View]
wait_for_raw_data (SUCCESS) -> run_spark_job (FAILED: OutOfMemory) -> validate_target (UPSTREAM_FAILED)
```

### The Clear Task Action (Slide 25):
* In legacy schedulers (Control-M, cron, SSIS), engineers frequently had to rerun the entire 4-hour batch from scratch.
* In Apache Airflow:
  1. Fix root cause (e.g. increase executor memory in config from `4g` to `8g`).
  2. In Airflow UI, click the failed task (`run_spark_job`) and click **Clear Task** (with *Downstream* selected).
  3. Airflow reruns only the failed step and subsequent downstream tasks, preserving the upstream work.


---
## Hands-on Lab Exercise 4 (Corresponding to Module 7 Hands-on)

### Task:
1. Add a new task `backup_raw_data` between `validate_target` and `notify_bi`.
2. Re-run `dag.test()` to verify that your new task executes in the correct order.


In [ ]:
# TODO: Create backup_raw_data task and wire it into the DAG:
# backup_raw_data = BashOperator(...)
# wait_for_raw_data >> run_spark_job >> validate_target >> backup_raw_data >> notify_bi

# dag.test(execution_date=datetime(2026, 9, 10))
